In [ ]:
"""
Stage 1 — Data Collection
Pulls 5 years of daily OHLCV data for Nifty50 stocks from NSE via yfinance.
Part of: Stock Return Prediction using LSTM (Quant Research roadmap, Tier 2)
"""

import yfinance as yf
import pandas as pd
import time
import os

# ── CONFIG ────────────────────────────────────────────────────────────────

# NSE tickers need a ".NS" suffix for yfinance. BSE would use ".BO" instead.
NIFTY50_SAMPLE = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS", "ICICIBANK.NS",
    "HINDUNILVR.NS", "ITC.NS", "SBIN.NS", "BHARTIARTL.NS", "KOTAKBANK.NS",
    "LT.NS", "AXISBANK.NS", "ASIANPAINT.NS", "MARUTI.NS", "TITAN.NS",
]

NIFTY50_INDEX = "^NSEI"   # Nifty 50 index itself — used as a market-wide feature

START_DATE = "2020-01-01"
END_DATE = "2025-06-19"   # adjust to "today" when you run this
OUTPUT_DIR = "data/raw"


def fetch_stock_data(ticker: str, start: str, end: str) -> pd.DataFrame:
    """
    Pull OHLCV data for a single ticker.
    auto_adjust=True (default in yfinance >=0.2.something) means Close is
    already split/dividend-adjusted — important so a stock split doesn't
    look like a -50% return crash in your training data.
    """
    df = yf.download(
        ticker,
        start=start,
        end=end,
        auto_adjust=True,    # adjusts OHLC for splits & dividends
        progress=False,
        multi_level_index=False,   # keeps columns flat: Open, High, Low, Close, Volume
    )

    if df.empty:
        print(f"  WARNING: no data returned for {ticker}")
        return df

    df["Ticker"] = ticker
    df.index.name = "Date"
    return df


def fetch_all(tickers: list[str], start: str, end: str, pause: float = 1.0) -> dict[str, pd.DataFrame]:
    """
    Pull data for a list of tickers with a small delay between calls
    to stay polite to Yahoo Finance's servers and avoid rate limiting.
    """
    data = {}
    for i, ticker in enumerate(tickers, 1):
        print(f"[{i}/{len(tickers)}] Fetching {ticker} ...")
        df = fetch_stock_data(ticker, start, end)
        if not df.empty:
            data[ticker] = df
        time.sleep(pause)
    return data


def save_data(data: dict[str, pd.DataFrame], output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    for ticker, df in data.items():
        # Strip ".NS" for a cleaner filename
        clean_name = ticker.replace(".NS", "").replace("^", "")
        path = os.path.join(output_dir, f"{clean_name}.csv")
        df.to_csv(path)
        print(f"  Saved {path}  ({len(df)} rows)")


if __name__ == "__main__":
    print("=" * 60)
    print("Fetching Nifty50 stock data + index benchmark")
    print("=" * 60)

    # 1. Fetch individual stocks
    stock_data = fetch_all(NIFTY50_SAMPLE, START_DATE, END_DATE)

    # 2. Fetch the Nifty50 index itself (your market-wide benchmark feature)
    print(f"\nFetching benchmark index {NIFTY50_INDEX} ...")
    index_data = {NIFTY50_INDEX: fetch_stock_data(NIFTY50_INDEX, START_DATE, END_DATE)}

    # 3. Save everything to disk
    print("\nSaving stock data...")
    save_data(stock_data, OUTPUT_DIR)

    print("\nSaving index data...")
    save_data(index_data, OUTPUT_DIR)

    # 4. Quick sanity check
    print("\n" + "=" * 60)
    print("Sanity check — RELIANCE.NS sample:")
    print("=" * 60)
    if "RELIANCE.NS" in stock_data:
        print(stock_data["RELIANCE.NS"].head())
        print(f"\nDate range: {stock_data['RELIANCE.NS'].index.min()} to {stock_data['RELIANCE.NS'].index.max()}")
        print(f"Total trading days: {len(stock_data['RELIANCE.NS'])}")

    print("\nDone. Data saved to:", OUTPUT_DIR)